# Занятие 28. Практика: логистическая регрессия и завершение курса

Это второе занятие по теме логистической регрессии: практическая работа в формате учебного рейтинга решений. Команды по 1–2 человека строят модель, сохраняют `submission.csv`, а преподаватель считает результат на скрытой test-выборке.

В блокноте уже есть рабочий стартовый baseline. Сначала запустите его и убедитесь, что получается `submission.csv`, затем улучшайте признаки, регуляризацию и порог.

**Задача:** предсказать, завершит ли ученик курс. Целевой столбец в train — `will_finish`.

**Главная метрика занятия:** F1-мера (F1-score). Больше — лучше.

F1 полезна здесь потому, что нам важны и найденные будущие “завершившие курс”, и количество ложных срабатываний.

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|---|---:|
| 1 | Validation и стратификация | 5 |
| 2 | Первый baseline | 6 |
| 3 | Подбор порога и вывод | 7 |
| 4 | Идеи для улучшения | 6 |
| 5 | Финальное обучение и submission | 6 |

## Правила и ориентир на 90 минут

1. 0–10 мин — понять данные, целевой класс и метрику.
2. 10–30 мин — запустить стартовый baseline логистической регрессии.
3. 30–60 мин — улучшить признаки и регуляризацию.
4. 60–80 мин — подобрать порог вероятности для F1.
5. 80–90 мин — сохранить submission и обсудить ошибки.

Используйте `test.csv` только для финального прогноза. Ответы к test не должны участвовать в выборе модели.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("competition") / "data"


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(train.shape, test.shape)
display(train.head())
display(sample_submission.head())


## Validation и стратификация — **5 баллов**

Стратификация нужна, чтобы доля классов 0 и 1 была похожей в train и validation.

## Первый baseline: вероятность → класс — **6 баллов**

Логистическая регрессия выдаёт вероятность класса 1. Чтобы получить класс, выбираем порог.

Если порог 0.5, то:

- вероятность 0.49 даёт класс 0;
- вероятность 0.51 даёт класс 1.

## Подбор порога — **7 баллов**

Модель может хорошо расставлять вероятности, но порог 0.5 не всегда лучший для F1.

Ниже мы перебираем пороги и выбираем тот, где F1 на validation максимальна.

## Идеи для улучшения — **6 баллов**

Попробуйте несколько вариантов и сравнивайте их на validation:

- `LogisticRegression(C=0.3)`, `C=1`, `C=3`;
- `class_weight="balanced"`;
- новые признаки:
  - `activity_total` — общий учебный ритм: часы практики + часть посещений клуба;
  - `score_per_missed_class` — последняя оценка с поправкой на число пропусков;
- проверить удаление слабого или дублирующего признака: убрать его, переобучить модель и сравнить F1 на той же validation-выборке;
- другой порог вероятности.

Не подбирайте порог по test: там нет честных ответов.

## Финальное обучение и submission — **6 баллов**

После выбора модели обучаем её на всём `train.csv` и делаем прогноз для `test.csv`.

## Что сдать

Сдайте файл `submission.csv`.

В нём должны быть столбцы:

- `id`;
- `probability` — вероятность класса 1;
- `will_finish` — итоговый прогноз 0 или 1.

Главная метрика считается по `will_finish`.